In [ ]:
from __future__ import annotations

import os
import pickle
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np

from tsfm_lens.utils import apply_custom_style

In [ ]:
apply_custom_style("../../config/plotting.yaml")

In [ ]:
model_name = "chronos"

In [ ]:
MODEL_LABELS = {
    "chronos": "Chronos",
    "chronos_bolt": "Chronos-Bolt",
    "timesfm": "TimesFM",
    "toto": "Toto",
}

ATTENTION_LABELS = {
    "ca": "Cross Attention",
    "sa": "Self Attention",
    "attn": "Attention",
}

ATTENTION_COLORS = {
    "ca": "C0",
    "sa": "C1",
    "attn": "C0",
}

ATTENTION_COLORS_ALT = {
    "ca": "lightcoral",
    "sa": "lightskyblue",
    "attn": "coral",
}


In [ ]:
model_label = MODEL_LABELS[model_name]

In [ ]:
def plot_entropic_rank(
    layers: np.ndarray,
    all_ranks: list[np.ndarray],
    median_rank: np.ndarray,
    title: str,
    ylabel: str,
    save_path: Path,
    attention_type: str,
    figsize: tuple[float, float] = (5, 5),
) -> None:
    fig, ax = plt.subplots(figsize=figsize)
    color = ATTENTION_COLORS.get(attention_type, "C0")

    # Plot individual curves
    for ranks in all_ranks:
        ax.plot(layers, ranks, color=color, alpha=0.1, linewidth=1)

    # Plot median
    ax.plot(
        layers,
        median_rank,
        color=color,
        linewidth=3,
        zorder=100,
        label=ATTENTION_LABELS.get(attention_type, attention_type.title()),
    )

    ax.set_xlabel("Layer", fontweight="bold")
    ax.set_ylabel(ylabel, fontweight="bold")
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(layers)
    ax.legend(frameon=True)

    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight")


def plot_combined_entropic_rank(
    layers: np.ndarray,
    median_by_type: dict[str, np.ndarray],
    ranks_by_type: dict[str, list[np.ndarray]],
    title: str,
    ylabel: str,
    save_path: Path,
    figsize: tuple[float, float] = (5, 5),
    legend_kwargs: dict[str, Any] = {},
    ylim: tuple[float, float] | None = None,
) -> None:
    fig, ax = plt.subplots(figsize=figsize)

    for attention_type, median in median_by_type.items():
        color = ATTENTION_COLORS.get(attention_type, None)
        lighter_color = ATTENTION_COLORS_ALT.get(attention_type, None)

        # Plot individual curves
        ranks_list = ranks_by_type.get(attention_type, [])
        for ranks in ranks_list:
            ax.plot(layers, ranks, color=lighter_color, alpha=0.03, linewidth=1)

        if ranks_list:
            ax.plot(layers, ranks, color="black", alpha=0.2, linewidth=1)
            ranks_array = np.array(ranks_list)
            p25 = np.percentile(ranks_array, 25, axis=0)
            p75 = np.percentile(ranks_array, 75, axis=0)
            ax.fill_between(layers, p25, p75, color=color, alpha=0.2)

        # Plot median
        ax.plot(
            layers,
            median,
            label=ATTENTION_LABELS.get(attention_type, attention_type.title()),
            linewidth=3,
            color=color,
            zorder=100,
        )

    ax.set_xlabel("Layer", fontweight="bold")
    ax.set_ylabel(ylabel, fontweight="bold")
    ax.set_title(title, fontweight="bold", fontsize=12)
    ax.set_xticks(layers)
    if ylim is not None:
        ax.set_ylim(*ylim)
    ax.legend(**legend_kwargs)

    fig.tight_layout()
    fig.savefig(save_path, bbox_inches="tight")


### Plotting

In [ ]:
work_dir = os.environ.get("WORK_DIR", "/work")
print(f"work_dir: {work_dir}")
outputs_dir = os.path.join(work_dir, "entropic_rank_v2")
print(f"outputs_dir: {outputs_dir}")
model_outputs_dir = os.path.join(outputs_dir, model_name)
print(f"model_outputs_dir: {model_outputs_dir}")

In [ ]:
full_dict_data_path = os.path.join(model_outputs_dir, "entropic_rank_data.pkl")
print(f"full_dict_data_path: {full_dict_data_path}")

with open(full_dict_data_path, "rb") as f:
    data = pickle.load(f)

per_attention_results = data["per_attention_results"]
layer_index_reference = data["layer_index_reference"]

In [ ]:
print(f"per_attention_results keys: {per_attention_results.keys()}")
print(f"layer_index_reference keys: {layer_index_reference.keys()}")

In [ ]:
attention_types = ["ca", "sa"]

In [ ]:
save_path = os.path.join("../../figs/head_outputs_metrics", f"{model_name}_combined_entropic_rank.pdf")
os.makedirs(os.path.dirname(save_path), exist_ok=True)
print(f"saving plot to: {save_path}")

if model_name in {"chronos", "chronos_bolt"}:
    required_types = [t for t in attention_types if t in {"ca", "sa"}]
    if all(attention_type in per_attention_results for attention_type in required_types):
        combined_layers = layer_index_reference[required_types[0]]
        median_by_type = {
            attention_type: np.median(np.stack(per_attention_results[attention_type], axis=0), axis=0)
            for attention_type in required_types
        }
        ranks_by_type = {attention_type: per_attention_results[attention_type] for attention_type in required_types}
        # title = f"Entropic Rank of Head Outputs ({model_label})"
        title = "Entropic Rank of Head Outputs"
        plot_combined_entropic_rank(
            layers=combined_layers,
            median_by_type=median_by_type,
            ranks_by_type=ranks_by_type,
            title=title,
            ylabel=r"Entropic Rank $\exp\left(H\right)$",
            save_path=save_path,
            figsize=(4, 6),
            legend_kwargs={"frameon": True},
            ylim=(None, 12.5),
        )